# Inference Optimization: GPU 추론 최적화 벤치마크

**목적**: `elmenwol/whisper-small_aihub_child` 모델의 GPU 추론 속도와 VRAM 효율을 극대화하기 위해, HuggingFace Pipeline을 제거하고 3가지 최적화 기법을 적용하여 속도·메모리 트레이드오프를 정량 비교합니다.

**최적화 기법 비교**:
| 기법 | 속도 (Speedup) | 모델 VRAM | 피크 VRAM | 특징 |
| --- | --- | --- | --- | --- |
| **Direct Inference** | 1.00x (6.02s) | 477.8 MB | 549.3 MB | Pipeline 제거, processor + model.generate() 직접 호출 |
| **Faster-Whisper** | 7.53x (0.80s) | 9.9 MB | 9.9 MB | CTranslate2 엔진, VRAM 98% 절감 |
| **torch.compile** | 9.45x (0.64s) | 487.6 MB | 549.2 MB | PyTorch 2.x 컴파일러, 최고 속도 |

**결론**:
- **최고 속도**: `torch.compile` (reduce-overhead) — 0.64s, 기준 대비 **9.5배 향상**
- **최소 VRAM**: `Faster-Whisper` (CTranslate2) — 모델 9.9MB, 피크 9.9MB로 **VRAM 98% 절감**
- **배포**: GPU 메모리 제한 환경(Cloud Run 등) → Faster-Whisper / 전용 GPU 서버 → torch.compile

---
**동시성 최적화 & CPU vs GPU 결론**:
- **CPU 한계**: GPU 단건 추론(0.39s)이 CPU(ONNX 12.75s)보다 **32.7배 빠르며**, 비용 효율 측면에서도 GPU가 **21배 저렴** (GCP Cloud Run 기준)
- **동시성 처리 (50건 기준)**:
  - `Baseline(순차)`: 가장 안정적이고 빠른 지연 시간 (QPS 5.28, P95 0.259s)
  - `num_workers`: 최고 처리량 (QPS 6.62)이나 꼬리 지연 심각 (P95 7.4s)
  - 직렬화 큐나 Micro-batch는 단일 GPU의 `faster-whisper`에서 병목으로 작용 (처리량 급감)
- **최종 결론**: 단일 GPU 환경에서는 외부 큐 없이 Baseline 순차 처리에 Rate limiting(요청 수 제한)을 거는 것이 최적

---

# CPU vs GPU Inference 비교 분석

- **환경**: Google Colab T4 GPU (15GB VRAM) · `elmenwol/whisper-small_aihub_child`

---

## 단건 추론 속도 비교

| 방법 | 시간 | vs CPU Direct | vs CPU Best | 메모리 |
|------|------|:---:|:---:|------|
| **CPU** Direct (FP32) | 16.70s | 1.00x | — | 2,516 MB (RSS) |
| **CPU** Dynamic Quant (INT8) | 65.16s | 0.26x | — | 3,693 MB (RSS) |
| **CPU** ONNX Runtime | 12.75s | 1.31x | 1.00x | 5,834 MB (RSS) |
| **GPU** Baseline (FP16+SDPA) | 0.39s | **42.8x** | **32.7x** | 549 MB (VRAM) |
| **GPU** Faster-Whisper (CT2) | 0.80s | 20.9x | 15.9x | 9.9 MB (VRAM) |
| **GPU** torch.compile | 0.64s | 26.1x | 19.9x | 549 MB (VRAM) |

- GPU Baseline은 CPU 대비 **최대 42.8배** 빠름. GPU에서 가장 느린 Faster-Whisper조차 CPU 최고 성능(ONNX)보다 **15.9배** 빠름.

---

## 처리량 비교 (N=50 동시 요청)

| 환경 | 방법 | QPS | 50건 총시간 | P95 지연 |
|------|------|:---:|:---:|:---:|
| CPU | Direct (순차) | ~0.06 | ~835s (추정) | ~16.7s |
| CPU | ONNX (순차) | ~0.08 | ~637s (추정) | ~12.7s |
| **GPU** | Baseline | **5.28** | **9.48s** | **0.259s** |
| **GPU** | num_workers | **6.62** | **7.55s** | 7.394s |

- CPU는 순차 처리만 가능하므로 50건 ≈ 50×12.7s = 635초.
- GPU Baseline은 동일 작업을 **9.48초**에 처리 — **67배 차이**.

---

## 메모리 효율 비교

```
CPU                                    GPU
┌─────────────────────────┐     ┌─────────────────────────┐
│ Direct    2,516 MB RAM  │     │ Direct     549 MB VRAM  │
│ INT8      3,693 MB RAM  │     │ CT2          10 MB VRAM │ ← 최소
│ ONNX      5,834 MB RAM  │     │ compile    549 MB VRAM  │
└─────────────────────────┘     └─────────────────────────┘
```

- CPU 최소 메모리 (FP32): **2,516 MB**
- GPU CT2 최소 메모리: **9.9 MB** — 254배 효율적
- GPU는 VRAM 외에 시스템 RAM도 사용하지만, 추론에 필요한 핵심 메모리는 극히 적음

---

## CPU 최적화의 한계

| CPU 기법 | 효과 | 문제점 |
|----------|------|--------|
| Dynamic Quantization (INT8) | **0.26x (역효과)** | Whisper 디코더의 attention이 INT8에 비적합, 오히려 4배 느림 |
| ONNX Runtime | **1.31x** | 속도 개선 미미, 메모리 2.3배 증가 (5,834MB) |

- CPU 최적화는 **최대 1.3배**가 한계. GPU 전환 시 **42.8배** 달성.

---

## 비용 효율 분석 (GCP Cloud Run 기준)

| 구성 | 비용/시간 | 50건 처리시간 | 비용/50건 | 효율 |
|------|-----------|:---:|-----------|------|
| CPU (4 vCPU, 8GB) | ~$0.00023/s | ~637s | ~$0.147 | 1x |
| GPU T4 (+ 4 vCPU) | ~$0.00075/s | ~9.5s | ~$0.007 | **21x** |

- GPU가 시간당 비용은 3.3배 비싸지만, 처리 속도 67배 차이로 **실제 비용은 GPU가 21배 저렴**.

---

## 결론


|  항목                |  CPU 최선        |  GPU 최선       |
|---------------------|-----------------|----------------|
|  단건 속도           |  12.75s (ONNX)  |  0.39s (FP16)  |
|  처리량 (50건)       |  ~0.08 QPS      |  5.28 QPS      |
|  메모리              |  2,516 MB       |  9.9 MB (CT2)  |
|  비용 효율           |  1x             |  21x           |


1. **CPU 추론은 프로토타이핑·개발 환경에서만 유효**. 프로덕션 서비스에는 부적합.
2. **GPU 전환만으로 42.8배 속도 향상** — 추가 최적화(torch.compile 등) 없이도 충분.
3. **VRAM 제한 환경**: Faster-Whisper (CT2)가 9.9MB로 운영 가능, L4/T4 공유 GPU에서 다중 모델 배포 가능.
4. **비용**: GPU가 시간당 비용은 높지만, 처리 속도 차이로 인해 **요청 단가는 GPU가 21배 저렴**.


## 0. 환경 설정 및 의존성 설치

In [ ]:
!pip install -q flash-attn --no-build-isolation
!pip install -q faster-whisper
!pip install -q librosa
!pip install -q ctranslate2

In [ ]:
import torch
import time
import librosa
import gc

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# 테스트 오디오 파일 경로 (Colab 환경에 맞게 수정)
AUDIO_FILE = '/content/서울의 중심에는 한강 하류가 동에서 .wav'
MODEL_NAME = 'elmenwol/whisper-small_aihub_child'

# 결과 저장용 (각 항목: {time, vram_model, vram_peak})
benchmark_results = {}

def get_gpu_memory_mb():
    """현재 GPU 메모리 사용량 (MB)"""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024**2
    return 0

def get_gpu_peak_memory_mb():
    """최대 GPU 메모리 사용량 (MB)"""
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / 1024**2
    return 0

def reset_gpu_stats():
    """GPU 메모리 통계 초기화"""
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()
        gc.collect()

---
## 1. Direct Inference: processor + model.generate() 직접 호출

HuggingFace `pipeline` 함수를 걷어내고, `WhisperProcessor`와 `model.generate()`를 직접 호출합니다.
Pipeline의 불필요한 오버헤드(입력 검증, 출력 후처리 래핑 등)를 제거하여 추론 속도를 개선합니다.

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# GPU 메모리 초기화
reset_gpu_stats()

# Processor 로드 (feature extractor + tokenizer 통합)
processor = WhisperProcessor.from_pretrained(MODEL_NAME)

# 모델 로드: FP16 + SDPA
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True,
    attn_implementation='sdpa',
)
model = model.to(device)
model.eval()

vram_model_direct = get_gpu_memory_mb()

print(f'Model dtype: {model.dtype}')
print(f'Device: {device}')
print(f'모델 로드 후 VRAM: {vram_model_direct:.1f} MB')

In [ ]:
def transcribe_direct(audio_file, model, processor, device, torch_dtype):
    """Pipeline 없이 processor + model.generate()로 직접 추론"""
    # 1. 오디오 로드
    audio_input, sr = librosa.load(audio_file, sr=16000)

    # 2. Feature extraction (processor로 입력 전처리)
    input_features = processor(
        audio_input,
        sampling_rate=16000,
        return_tensors='pt'
    ).input_features.to(device=device, dtype=torch_dtype)

    # 3. 한국어 전사를 위한 forced_decoder_ids 설정
    forced_decoder_ids = processor.get_decoder_prompt_ids(
        language='ko',
        task='transcribe'
    )

    # 4. 추론 수행
    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            forced_decoder_ids=forced_decoder_ids,
            max_new_tokens=255,
        )

    # 5. 디코딩
    transcription = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]
    return transcription

In [ ]:
# 벤치마크: Direct Inference
reset_gpu_stats()
# 모델은 이미 로드되어 있으므로 현재 사용량부터 기록
torch.cuda.reset_peak_memory_stats()

start_time = time.time()
transcription = transcribe_direct(AUDIO_FILE, model, processor, device, torch_dtype)
elapsed = time.time() - start_time
vram_peak_direct = get_gpu_peak_memory_mb()

print(f'[Direct Inference]')
print(f'  결과: {transcription}')
print(f'  소요 시간: {elapsed:.4f}초')
print(f'  모델 VRAM: {vram_model_direct:.1f} MB')
print(f'  추론 시 피크 VRAM: {vram_peak_direct:.1f} MB')

benchmark_results['Direct Inference (processor + model.generate)'] = {
    'time': elapsed,
    'vram_model': vram_model_direct,
    'vram_peak': vram_peak_direct,
}

# 다음 벤치마크를 위해 모델 해제
del model
torch.cuda.empty_cache()
gc.collect()

---
## 2. Faster-Whisper: CTranslate2 기반 추론 엔진

모델 가중치는 동일하게 유지하면서, 추론 엔진만 CTranslate2 기반의 `faster-whisper`로 교체합니다.

HuggingFace 모델은 CTranslate2 포맷(`model.bin`)이 아니므로, 먼저 변환한 뒤 로드합니다.
이 모델은 `tokenizer.json`이 없으므로, processor에서 fast tokenizer를 생성하여 변환 디렉토리에 저장합니다.

**장점**:
- 추론 속도 7.5배 향상 (0.80s)
- VRAM 사용량 98% 감소 (9.9MB)
- 코드 변경 최소화

In [ ]:
import os
import shutil
import ctranslate2
from huggingface_hub import hf_hub_download

# HuggingFace Whisper 모델 → CTranslate2 포맷 변환
CT2_OUTPUT_DIR = '/content/whisper-small-aihub-ct2'

if not os.path.exists(os.path.join(CT2_OUTPUT_DIR, 'model.bin')):
    print('CTranslate2 포맷으로 변환 중...')

    # 1. 모델 가중치 변환 (copy_files 없이 변환만 수행)
    converter = ctranslate2.converters.TransformersConverter(MODEL_NAME)
    converter.convert(
        output_dir=CT2_OUTPUT_DIR,
        quantization='float16',
        force=True,
    )

    # 2. preprocessor_config.json 복사 (feature extractor 설정)
    preprocessor_path = hf_hub_download(MODEL_NAME, 'preprocessor_config.json')
    shutil.copy(preprocessor_path, os.path.join(CT2_OUTPUT_DIR, 'preprocessor_config.json'))

    # 3. tokenizer.json 생성 (이 모델은 slow tokenizer만 있으므로 fast tokenizer로 변환)
    processor.tokenizer.save_pretrained(CT2_OUTPUT_DIR)

    print(f'변환 완료: {CT2_OUTPUT_DIR}')
    print(f'생성된 파일: {os.listdir(CT2_OUTPUT_DIR)}')
else:
    print(f'이미 변환된 모델 존재: {CT2_OUTPUT_DIR}')

In [ ]:
from faster_whisper import WhisperModel

reset_gpu_stats()

# 변환된 CTranslate2 모델로 Faster-Whisper 로드
faster_model = WhisperModel(
    CT2_OUTPUT_DIR,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    compute_type='float16' if torch.cuda.is_available() else 'float32',
)

vram_model_fw = get_gpu_memory_mb()
print(f'Faster-Whisper 모델 로드 완료')
print(f'모델 로드 후 VRAM: {vram_model_fw:.1f} MB')

In [ ]:
def transcribe_faster_whisper(audio_file, model):
    """Faster-Whisper (CTranslate2)로 추론"""
    segments, info = model.transcribe(
        audio_file,
        language='ko',
        task='transcribe',
        beam_size=5,
    )

    # 세그먼트를 하나의 텍스트로 결합
    transcription = ''.join([segment.text for segment in segments])
    return transcription

In [ ]:
# 벤치마크: Faster-Whisper
torch.cuda.reset_peak_memory_stats()

start_time = time.time()
transcription_fw = transcribe_faster_whisper(AUDIO_FILE, faster_model)
elapsed_fw = time.time() - start_time
vram_peak_fw = get_gpu_peak_memory_mb()

print(f'[Faster-Whisper]')
print(f'  결과: {transcription_fw}')
print(f'  소요 시간: {elapsed_fw:.4f}초')
print(f'  모델 VRAM: {vram_model_fw:.1f} MB')
print(f'  추론 시 피크 VRAM: {vram_peak_fw:.1f} MB')

benchmark_results['Faster-Whisper (CTranslate2)'] = {
    'time': elapsed_fw,
    'vram_model': vram_model_fw,
    'vram_peak': vram_peak_fw,
}

# 다음 벤치마크를 위해 모델 해제
del faster_model
torch.cuda.empty_cache()
gc.collect()

---
## 3. torch.compile 적용

PyTorch 2.x의 `torch.compile`을 모델에 적용하여 디코딩 속도를 높입니다.
첫 번째 추론은 컴파일 오버헤드가 있으므로, 웜업 후 실제 속도를 측정합니다.

In [ ]:
reset_gpu_stats()

# 기존 HuggingFace 모델을 다시 로드하고 torch.compile 적용
model_compiled = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True,
    attn_implementation='sdpa',
)
model_compiled = model_compiled.to(device)
model_compiled.eval()

# torch.compile 적용
model_compiled = torch.compile(model_compiled, mode='reduce-overhead')

vram_model_tc = get_gpu_memory_mb()
print(f'torch.compile 적용 완료 (mode=reduce-overhead)')
print(f'모델 로드 후 VRAM: {vram_model_tc:.1f} MB')

In [ ]:
# 웜업 (첫 번째 추론은 컴파일 오버헤드 포함)
print('Warming up torch.compile...')
_ = transcribe_direct(AUDIO_FILE, model_compiled, processor, device, torch_dtype)
print('웜업 완료')

In [ ]:
# 벤치마크: torch.compile
torch.cuda.reset_peak_memory_stats()

start_time = time.time()
transcription_tc = transcribe_direct(AUDIO_FILE, model_compiled, processor, device, torch_dtype)
elapsed_tc = time.time() - start_time
vram_peak_tc = get_gpu_peak_memory_mb()

print(f'[torch.compile + Direct Inference]')
print(f'  결과: {transcription_tc}')
print(f'  소요 시간: {elapsed_tc:.4f}초')
print(f'  모델 VRAM: {vram_model_tc:.1f} MB')
print(f'  추론 시 피크 VRAM: {vram_peak_tc:.1f} MB')

benchmark_results['torch.compile (reduce-overhead)'] = {
    'time': elapsed_tc,
    'vram_model': vram_model_tc,
    'vram_peak': vram_peak_tc,
}

del model_compiled
torch.cuda.empty_cache()
gc.collect()

---
## 4. 벤치마크 결과 비교

추론 속도와 배포 효율(VRAM 사용량)을 종합 비교합니다.

In [ ]:
print('=' * 90)
print(f'{"Method":<45} {"Time":>8} {"Speedup":>8} {"Model":>10} {"Peak":>10}')
print(f'{"":<45} {"(sec)":>8} {"":>8} {"VRAM(MB)":>10} {"VRAM(MB)":>10}')
print('-' * 90)

baseline_time = list(benchmark_results.values())[0]['time']
for method, data in benchmark_results.items():
    speedup = baseline_time / data['time'] if data['time'] > 0 else float('inf')
    print(f'{method:<45} {data["time"]:>7.4f}s {speedup:>7.2f}x {data["vram_model"]:>9.1f} {data["vram_peak"]:>9.1f}')

print('=' * 90)

# 최고 성능 요약
fastest = min(benchmark_results, key=lambda k: benchmark_results[k]['time'])
lowest_vram = min(benchmark_results, key=lambda k: benchmark_results[k]['vram_peak'])

print(f'\n⚡ 최고 속도: {fastest}')
print(f'   → {benchmark_results[fastest]["time"]:.4f}s ({baseline_time / benchmark_results[fastest]["time"]:.1f}x faster)')

print(f'\n💾 최소 VRAM: {lowest_vram}')
print(f'   → 모델: {benchmark_results[lowest_vram]["vram_model"]:.1f} MB, 피크: {benchmark_results[lowest_vram]["vram_peak"]:.1f} MB')

print(f'\n📋 배포 추천:')
print(f'   - GPU 메모리 제한 환경 (Cloud Run 등) → {lowest_vram}')
print(f'   - 전용 GPU 서버 (최저 지연시간)       → {fastest}')